# 11 · Spatial pattern, uncertainty, and scale


**Mission (40 minutes):** calculate global Moran's I, use a seeded permutation reference distribution, and show how aggregation changes a map. This is exploratory statistics on 12 fictional sectors, not an outbreak detector.


**Runtime:** Python + NumPy + Matplotlib; no model key. Run all cells in order. Interactive viewers additionally load JavaScript from a CDN. All scenario records are fictional.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from gev_lab import *
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
scenario = styled_sectors()
stats = sector_summary()
print('SYNTHETIC TRAINING EXERCISE | snapshot', AS_OF)

## 1. Define “neighbor” before computing a pattern
The rectangles form a 3 × 4 grid. Rook neighbors share an edge. We use **symmetric binary weights**, with no diagonal self-neighbor, and a two-sided permutation comparison centered on the randomization expectation −1/(n−1).

In [ ]:
values=np.array([s['rate_10000'] for s in stats]);w=rook_weights()
observed=moran_i(values,w);expected=-1/(len(values)-1)
rng=np.random.default_rng(20261002)
permuted=np.array([moran_i(rng.permutation(values),w) for _ in range(999)])
p=(1+np.count_nonzero(np.abs(permuted-expected)>=abs(observed-expected)))/(len(permuted)+1)
fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
axes[0].imshow(w,cmap='Blues');axes[0].set(title='Binary rook adjacency',xlabel='Sector index',ylabel='Sector index')
axes[1].hist(permuted,bins=30,color='#42b8ad',edgecolor='white');axes[1].axvline(observed,color='#ad442e',label=f'Observed I={observed:.3f}')
axes[1].axvline(expected,color='gray',linestyle='--',label='Randomization expectation')
axes[1].set(xlabel="Global Moran's I",ylabel='Permutations',title=f'Two-sided permutation p ≈ {p:.3f}');axes[1].legend();plt.show()

A small permutation p-value is conditional on the chosen rates, weights, and exchangeability assumption. It is not the probability that an outbreak exists and does not identify a local hotspot. Unequal populations produce unequal precision, weakening simple rate-exchangeability assumptions.

## 2. Aggregation changes the visible pattern (MAUP)
Aggregate counts and denominators **before** dividing. The unweighted mean of sector rates generally differs from a pooled rate.

In [ ]:
counts=np.array([s['cases'] for s in stats]).reshape(3,4)
pop=np.array([s['population'] for s in stats]).reshape(3,4)
pooled_rows=rate_per_10000(counts.sum(axis=1),pop.sum(axis=1))
pooled_cols=rate_per_10000(counts.sum(axis=0),pop.sum(axis=0))
fig,axes=plt.subplots(1,2,figsize=(10,4),constrained_layout=True)
axes[0].bar(range(3),pooled_rows,color='#169c96');axes[0].set(xlabel='Row grouping',ylabel='Reported cases / 10,000',title='Same reports, 3 regions')
axes[1].bar(range(4),pooled_cols,color='#d79536');axes[1].set(xlabel='Column grouping',ylabel='Reported cases / 10,000',title='Same reports, 4 regions')
upper=max(pooled_rows.max(),pooled_cols.max())*1.15
for ax in axes: ax.set_ylim(0,upper)
plt.show()
print('Pooled rate:',float(rate_per_10000(counts.sum(),pop.sum())))
print('Unweighted mean sector rate:',values.mean())

## Lab challenge
Use queen adjacency (edge or corner) and compare I. Shuffle the rates; do you always get zero? Explain why selecting whichever weights gives the lowest p-value is misleading. Discuss ecological fallacy and why privacy protection needs more than hiding small numbers on a map.

**Check:** both row and column pooling conserve the total count and population. No local significance or causal conclusion follows from this global statistic.

**Sources:** [PySAL Moran reference](https://pysal.org/esda/stable/user-guide/global_morans_i.html) for the statistic and weighting conventions; [CDC analysis and interpretation](https://www.cdc.gov/field-epi-manual/php/chapters/analyze-interpret-data.html). The implementation is original NumPy and does not require PySAL in the browser.